# 02 — Plus de flexibilité aide-t-elle ?

**Question :** des arbres font-ils mieux que le meilleur modèle linéaire du notebook 01 (PDQ + mois + t + t², MAE 5,99) ?
Le modèle linéaire donne la même tendance à tous les PDQ : il sous-estime les gros postes.
Un arbre peut apprendre une tendance différente par PDQ, mais il risque d'apprendre le train par cœur.

On compare sur la **validation (2023)** : l'erreur moyenne, les grosses erreurs (p90), l'erreur sur les **5 plus gros PDQ**
et l'écart entre train et validation. Le **test reste fermé** jusqu'au choix final.

**Contrat (v1, inchangé) :** pour chaque PDQ actif, avant le début de la semaine, estimer les actes déclarés de la semaine,
à partir du calendrier seulement (PDQ, mois, temps écoulé).

In [ ]:
import json
import hashlib
import tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)

SEED = cfg["experiment"]["seed"]
DATA_PATH = ROOT / cfg["data"]["path"]
IN = ROOT / "artifacts" / "00_start_here"

panel = pd.read_parquet(IN / "panel.parquet")
meta = json.loads((IN / "meta.json").read_text())
meta_01 = json.loads((ROOT / "artifacts" / "01_regression_lineaire" / "meta.json").read_text())

empreinte = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:16]
assert empreinte == meta["source_sha16"], "Le CSV a changé depuis le notebook 00 : relance le 00 d'abord."
assert meta_01["source_sha16"] == meta["source_sha16"], "Le notebook 01 a été fait sur d'autres données : relance le 01."

panel["t"] = (panel["SEMAINE"] - panel["SEMAINE"].min()).dt.days / 365.25
panel["t2"] = panel["t"] ** 2
panel["mois"] = panel["SEMAINE"].dt.month

print("Panel :", panel.shape)
print("À battre (notebook 01) :", meta_01["modele_retenu"], "| MAE validation", meta_01["mae_validation"])

## Choix fixés avant d'entraîner
Métrique principale : MAE. Tranche de décision : les 5 PDQ les plus chargés du **train**,
là où une erreur coûte le plus en effectifs, et là où le modèle linéaire sous-estime.
Elle est choisie **avant** de voir les résultats des arbres, pour ne pas choisir la tranche qui arrange.

In [ ]:
METRIQUE = "mae"

moyennes_pdq = panel[panel["ENSEMBLE"] == "train"].groupby("PDQ")["y"].mean()
GROS_PDQ = moyennes_pdq.nlargest(5).index.tolist()

print("Tranche de décision (5 plus gros PDQ du train) :", GROS_PDQ)
moyennes_pdq.loc[GROS_PDQ].round(1)

## Outil d'évaluation
Même fonction que le notebook 01, avec trois mesures de plus :
- `p90_erreur` : 90 % des erreurs sont plus petites que cette valeur (la taille des grosses erreurs) ;
- `mae_gros_pdq` : l'erreur moyenne sur les 5 plus gros PDQ ;
- `biais_gros_pdq` : la moyenne de l'erreur avec son signe sur ces PDQ (positif = sous-estimés).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

train = panel[panel["ENSEMBLE"] == "train"]
validation = panel[panel["ENSEMBLE"] == "validation"]
# Pas de variable « test » : il reste fermé jusqu'à la fin du notebook.

modeles = {}
resultats = []

def evaluer(nom, estimateur, numeriques=(), categorielles=()):
    colonnes = list(numeriques) + list(categorielles)

    # 1. Préparation des colonnes (identique au notebook 01)
    etapes = []
    if numeriques:
        etapes.append(("num", StandardScaler(), list(numeriques)))
    if categorielles:
        etapes.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(categorielles)))

    # 2. Entraînement sur le TRAIN seulement
    modele = make_pipeline(ColumnTransformer(etapes), estimateur)
    modele.fit(train[colonnes], train["y"])

    # 3. Prédictions et erreurs
    pred_validation = modele.predict(validation[colonnes])
    pred_train = modele.predict(train[colonnes])
    erreur = validation["y"] - pred_validation
    gros = validation["PDQ"].isin(GROS_PDQ)

    modeles[nom] = modele
    resultats.append({
        "modele": nom,
        "mae_validation": mean_absolute_error(validation["y"], pred_validation),
        "rmse_validation": root_mean_squared_error(validation["y"], pred_validation),
        "mae_train": mean_absolute_error(train["y"], pred_train),
        "p90_erreur": erreur.abs().quantile(0.9),
        "mae_gros_pdq": erreur[gros].abs().mean(),
        "biais_gros_pdq": erreur[gros].mean(),
    })
    return pd.DataFrame(resultats).set_index("modele").round(3)

## Références
On refait la constante et le meilleur modèle linéaire du notebook 01, avec la même fonction, pour comparer les arbres à armes égales.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

evaluer("moyenne", DummyRegressor(strategy="mean"), numeriques=["t"])
evaluer("mediane", DummyRegressor(strategy="median"), numeriques=["t"])
evaluer("lineaire_t2", LinearRegression(), numeriques=["t", "t2"], categorielles=["PDQ", "mois"])

## Un arbre peu profond
Au plus 5 questions (32 feuilles), au moins 10 semaines par feuille.
Hypothèse : en posant des questions sur le PDQ **puis** sur le temps, l'arbre peut donner une tendance différente aux gros PDQ.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

evaluer("arbre_court", DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=SEED),
        numeriques=["t"], categorielles=["PDQ", "mois"])

## Un arbre sans limite
Aucune limite de profondeur, et une feuille peut ne contenir qu'une seule semaine.
Prédiction à vérifier : que deviennent l'erreur sur le train et l'erreur sur la validation ?

In [ ]:
evaluer("arbre_profond", DecisionTreeRegressor(max_depth=None, min_samples_leaf=1, random_state=SEED),
        numeriques=["t"], categorielles=["PDQ", "mois"])

## Une forêt : la moyenne de 100 arbres
Chaque arbre apprend sur un tirage au hasard des semaines du train ; la forêt fait la moyenne de leurs prédictions.
Les erreurs propres à chaque arbre se compensent en partie.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

evaluer("foret", RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=3,
                                       max_features=1.0, random_state=SEED, n_jobs=1),
        numeriques=["t"], categorielles=["PDQ", "mois"])

## Le boosting : des petits arbres qui corrigent les erreurs des précédents
Un premier petit arbre prédit ; le suivant apprend ses erreurs ; le suivant apprend les erreurs restantes, etc.
Chaque arbre ne corrige qu'une petite part (8 %) de ce qui reste, en 120 étapes.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

evaluer("boosting", HistGradientBoostingRegressor(max_iter=120, max_leaf_nodes=15, learning_rate=0.08,
                                                  min_samples_leaf=20, early_stopping=False, random_state=SEED),
        numeriques=["t"], categorielles=["PDQ", "mois"])

## Écart entre train et validation
Un grand écart signale un modèle qui a appris le train par cœur.

In [ ]:
ecarts = pd.DataFrame(resultats).set_index("modele")[["mae_train", "mae_validation"]]
ecarts["ecart"] = ecarts["mae_validation"] - ecarts["mae_train"]
ecarts.round(3)

**Constat — Flexibilité.** Un arbre court (7,40) est trop grossier : 32 feuilles pour 29 PDQ, les mois et les années.
Sans limite, l'arbre apprend le train par cœur (MAE train 0,00) et reste moins bon que le linéaire en validation (6,62).
La forêt égale le linéaire (6,01) sans corriger les gros PDQ (biais +5,0).
Le **boosting** est le meilleur sur toutes les mesures : MAE 5,17, p90 11,4, MAE gros PDQ 7,8, avec un petit écart train/validation (0,86).
Il réduit le biais des gros PDQ (+3,7 contre +4,5) sans le supprimer.